In [ ]:
# 各种import
import os
import torch
import shutil
import seaborn as sns
from tqdm import tqdm
import torch.nn as nn
import matplotlib.pyplot as plt
from model import resnet
from conf import cal_accuracy
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print(torch.version.cuda)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

In [ ]:
# 加载模型
model = resnet(channels=[64, 64, 128, 256, 128]).to(device)
model.load_state_dict(torch.load('./best_model/best_model.pth'))
model.eval()

In [ ]:
conv_layers = 0
fc_layers = 0

for module in model.modules():
    if isinstance(module, nn.Conv2d):
        conv_layers += 1
    elif isinstance(module, nn.Linear):
        fc_layers += 1

print(f"Conv: {conv_layers}")
print(f"Linear: {fc_layers}")

In [ ]:
# 图象转为tensor && 像素颜色归一化
transform = transforms.Compose([
                                transforms.Resize(224), 
                                transforms.CenterCrop(224),
                                transforms.ToTensor(), 
                                transforms.Normalize(mean=[0.5, ], std=[0.5, ])])
testset= datasets.Flowers102(root="./flowers", download=True, transform=transform, split='train')
test_loader = DataLoader(testset, batch_size=256, shuffle=True)

In [ ]:
model.eval()
with torch.no_grad():
    criterion = nn.CrossEntropyLoss().to(device)
    correct = 0
    total = 0
    y_test, y_pred, output_all = torch.tensor([]).to(device), torch.tensor([]).to(device), torch.tensor([]).to(device)
    for data, target in tqdm(test_loader):
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output, 1)
        y_test = torch.concat([y_test, target])
        y_pred = torch.concat([y_pred, predicted])
        output_all = torch.concat([output_all, output])
        correct += (predicted == target).sum().item()
        total += target.size(0)
    accuracy = correct/total
    loss = criterion(output_all, y_test.long())
    # y_test, y_pred = y_test.to('cpu'), y_pred.to('cpu')
    # print(classification_report(y_test, y_pred))

In [ ]:
y_test, y_pred = y_test.to('cpu'), y_pred.to('cpu')
print(f"loss:{loss}, accuracy:{accuracy}")
print(classification_report(y_test.long(), y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, fmt="d", cmap="Blues")
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()